# Project Workflow Overview:
1. Import Libraries
2. Load DataFrame
3. Clean DataFrame
    <br>
    3.1 Convert Time Format
    <br>
    3.2 Clean **Actor** Field
    <br> 
    3.3 Clean **Filename** Field 
4. Anonymize Data

## Import Libraries

In [1]:
import sys
sys.path.append('../') # these two lines allow the notebook to find the path to the source code contained in 'src'
import pandas as pd
import re
import io_utils, data_cleaning, data_anonymization
#from tests import test_preprocessing, test_anonymization
from src.data.constants import INTERIM_DATA_DIR

## Load DataFrame
- Use functions in file **io_utils.py.py**
- The csv file is the output of process_raw_data

In [2]:
df = io_utils.reading_dataframe(dir= INTERIM_DATA_DIR, file_name='traces250102_clean.csv')

Directory is ok.


### Print 5 rows and columns of dataframe

In [3]:
df.head()

,research_usage,result,_id.$oid,timestamp.$date,stored.$date,actor,actor.objectType,verb,object,object.objectType,...,P_codeState,filename,F_codeState,Debug_TimeStampEnd,Debug_TimeStampActions,tests,function,time_delta,session.duration,seance
0,1.0,NaN,675aa485a7dff6d2d4811b45,2024-12-12 09:53:24.624000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7 days 00:05:05.313000,NaN,CTP
1,1.0,NaN,675aa4cea7dff6d2d4811b4f,2024-12-12 09:53:25.008000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,Activity,...,NaN,/home/l1/abaly.oura.etu/Downloads/puissance4.py,"# Compléter ici (noms, groupe, contenu fichier...",NaN,NaN,NaN,NaN,0 days 00:00:00.384000,NaN,CTP
2,1.0,NaN,675aa4cea7dff6d2d4811b50,2024-12-12 09:54:38.793000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.End,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:01:13.785000,0 days 00:01:14.169000,CTP
3,1.0,NaN,671b54aabd5a98b8f9dc8c14,2024-10-25 10:19:54.344000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1 days 00:21:30.977000,NaN,semaine_8
4,1.0,NaN,671b54f6bd5a98b8f9dc8c45,2024-10-25 10:19:54.583000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,Activity,...,NaN,/home/l1/abaly.oura.etu/TDM 8/devinette.py,# L1 MI S1 - Prog\n# TP exos de manipulation -...,NaN,NaN,NaN,NaN,0 days 00:00:00.239000,NaN,semaine_8


## Clean DataFrame

### Convert Time Format

In [4]:
# Before
df[['session.id', 'timestamp.$date', 'time_delta', 'session.duration']]

,session.id,timestamp.$date,time_delta,session.duration
0,139682817762848,2024-12-12 09:53:24.624000+00:00,7 days 00:05:05.313000,NaN
1,139682817762848,2024-12-12 09:53:25.008000+00:00,0 days 00:00:00.384000,NaN
2,139682817762848,2024-12-12 09:54:38.793000+00:00,0 days 00:01:13.785000,0 days 00:01:14.169000
3,139779362711072,2024-10-25 10:19:54.344000+00:00,1 days 00:21:30.977000,NaN
4,139779362711072,2024-10-25 10:19:54.583000+00:00,0 days 00:00:00.239000,NaN
...,...,...,...,...
306941,139911616626448,2024-09-04 14:16:21.990000+00:00,0 days 00:00:53.790000,NaN
306942,139911616626448,2024-09-04 14:18:08.724000+00:00,0 days 00:01:46.734000,NaN
306943,139911616626448,2024-09-04 14:18:08.725000+00:00,0 days 00:00:00.001000,0 days 00:22:32.511000
306944,140578214641424,2024-09-04 13:29:54.300000+00:00,0 days 00:00:00,NaN


In [5]:
# Apply
df = data_cleaning.clean_time(df)

In [6]:
# After
df[['session.id', 'timestamp.$date', 'time_delta', 'session.duration']]

,session.id,timestamp.$date,time_delta,session.duration
0,139682817762848,2024-12-12 09:53:24.624000+00:00,7 days 00:05:05.313000,NaN
1,139682817762848,2024-12-12 09:53:25.008000+00:00,0 days 00:00:00.384000,NaN
2,139682817762848,2024-12-12 09:54:38.793000+00:00,0 days 00:01:13.785000,0 days 00:01:14.169000
3,139779362711072,2024-10-25 10:19:54.344000+00:00,1 days 00:21:30.977000,NaN
4,139779362711072,2024-10-25 10:19:54.583000+00:00,0 days 00:00:00.239000,NaN
...,...,...,...,...
306941,139911616626448,2024-09-04 14:16:21.990000+00:00,0 days 00:00:53.790000,NaN
306942,139911616626448,2024-09-04 14:18:08.724000+00:00,0 days 00:01:46.734000,NaN
306943,139911616626448,2024-09-04 14:18:08.725000+00:00,0 days 00:00:00.001000,0 days 00:22:32.511000
306944,140578214641424,2024-09-04 13:29:54.300000+00:00,0 days 00:00:00,NaN


### Clean **Actor** Field
Process: 
1. Extract all the non matching name of actor with prenom.nom.etu -> **incorrect = not_a_correct_identifier()**
2. Delete emails at the end -> **df = delete_end_email()**
3. Delete all rows of nebut -> **df = delete_actor_lines()**
4. Split actor and binome into 2 columns -> **df = split_actor_binome()**
5. Remove actors or binomes name -> **df = delete_name_actor_binome()**
6. Replace None value by " " -> **df = replace_None_by_str()**
7. Replace joker -> **df = replace_jokers()**
8. Cleaning manually -> **df = cleaning_manual_actors_2425()**

#### 1. Extract all the non matching name of actor with prenom.nom.etu

In [7]:
incorrect = data_cleaning.not_a_correct_identifier(df)
incorrect

0                                   abaly.oura.etu/
1                     abaly.oura.etu@univ-lille.fr/
2                      abdallah.saint-ghislain.etu/
3                    abdelrahmane.bendjeladjel.etu/
4                            abderrahmen.ayadi.etu/
                           ...                     
527                             youssef.airaki.etu/
528                               yves.djegnon.etu/
529    yves.djegnon.etu/fatoumata-bintou.traore.etu
530                             zacharie.deroo.etu/
531           zacharie.deroo.etu/noe.carpentier.etu
Length: 532, dtype: object

#### 2. Delete the email at the end

In [8]:
## Before deleting
print(df['actor'].unique())

['abaly.oura.etu/' 'abaly.oura.etu@univ-lille.fr/'
 'abdallah.saint-ghislain.etu/' 'abdelrahmane.bendjeladjel.etu/'
 'abderrahmen.ayadi.etu/' 'abderrahmen.ayadi.etu/abderrahmen.ayadi.etu'
 'abderrahmen.ayadi.etu/hazem.hlioui.etu' 'abdoulaye.mbaye.etu/'
 'abdoulaye.nguere.etu/' 'abdoulaye.nguere.etu/serigne.cisse.etu'
 'adam.chikh-touami.etu/' 'adam.chikh-touami.etu/ilian.briki.etu'
 'adam.oukzedou.etu/' 'adam.sialiti.etu/' 'adama-eliya.avlah.etu/'
 'adama-eliya.avlah.etu/edohson-arnaud.guedou.etu' 'adame.chairi.etu/'
 'adem.kaloun.etu/' 'adem.kaloun.etu/yasser.jad.etu'
 'adnane.el-mazouni.etu/'
 'adnane.el-mazouni.etu/nils.charfeddine-barbier.etu' 'adrien.vetter.etu/'
 'ahmed.bouziani.etu/' 'ahmed.bouziani.etu/nassim.belouar.etu'
 'ahmed.terfas.etu/rayane.denni.etu' 'ahmedzeidane.mohamedelmoctar.etu/'
 'aissatou-bobo.diallo.etu/'
 'aissatou-bobo.diallo.etu/aissatou-bobo.diallo.etu'
 'aissatou-bobo.diallo.etu/fadil.sani-labo.etu'
 'alban.gahide-fournier.etu/' 'alban.gahide-fournier.etu/

In [9]:
# Apply deleting
df = data_cleaning.delete_end_email(df)
df['actor'].unique() # After deleting

array(['abaly.oura.etu/', 'abaly.oura.etu',
       'abdallah.saint-ghislain.etu/', 'abdelrahmane.bendjeladjel.etu/',
       'abderrahmen.ayadi.etu/',
       'abderrahmen.ayadi.etu/abderrahmen.ayadi.etu',
       'abderrahmen.ayadi.etu/hazem.hlioui.etu', 'abdoulaye.mbaye.etu/',
       'abdoulaye.nguere.etu/', 'abdoulaye.nguere.etu/serigne.cisse.etu',
       'adam.chikh-touami.etu/', 'adam.chikh-touami.etu/ilian.briki.etu',
       'adam.oukzedou.etu/', 'adam.sialiti.etu/',
       'adama-eliya.avlah.etu/',
       'adama-eliya.avlah.etu/edohson-arnaud.guedou.etu',
       'adame.chairi.etu/', 'adem.kaloun.etu/',
       'adem.kaloun.etu/yasser.jad.etu', 'adnane.el-mazouni.etu/',
       'adnane.el-mazouni.etu/nils.charfeddine-barbier.etu',
       'adrien.vetter.etu/', 'ahmed.bouziani.etu/',
       'ahmed.bouziani.etu/nassim.belouar.etu',
       'ahmed.terfas.etu/rayane.denni.etu',
       'ahmedzeidane.mohamedelmoctar.etu/', 'aissatou-bobo.diallo.etu/',
       'aissatou-bobo.diallo.etu/aissatou

#### 3. Delete all rows of nebut

In [10]:
# # Before
print(len(df[df['actor'] == 'nebut/'])) 

# Apply
df = data_cleaning.delete_actor_lines(df, "nebut/")

# After
print(len(df[df['actor'] == 'nebut/'])) 

30
0


####  4. Split actor and binome into 2 columns

In [11]:
# Before
df['actor'].head(50)

0     abaly.oura.etu/
1     abaly.oura.etu/
2     abaly.oura.etu/
3     abaly.oura.etu/
4     abaly.oura.etu/
5     abaly.oura.etu/
6     abaly.oura.etu/
7     abaly.oura.etu/
8     abaly.oura.etu/
9     abaly.oura.etu/
10    abaly.oura.etu/
11    abaly.oura.etu/
12    abaly.oura.etu/
13    abaly.oura.etu/
14    abaly.oura.etu/
15    abaly.oura.etu/
16    abaly.oura.etu/
17    abaly.oura.etu/
18    abaly.oura.etu/
19    abaly.oura.etu/
20    abaly.oura.etu/
21    abaly.oura.etu/
22    abaly.oura.etu/
23    abaly.oura.etu/
24    abaly.oura.etu/
25    abaly.oura.etu/
26    abaly.oura.etu/
27    abaly.oura.etu/
28    abaly.oura.etu/
29    abaly.oura.etu/
30    abaly.oura.etu/
31    abaly.oura.etu/
32    abaly.oura.etu/
33    abaly.oura.etu/
34    abaly.oura.etu/
35    abaly.oura.etu/
36    abaly.oura.etu/
37    abaly.oura.etu/
38    abaly.oura.etu/
39    abaly.oura.etu/
40    abaly.oura.etu/
41    abaly.oura.etu/
42    abaly.oura.etu/
43    abaly.oura.etu/
44    abaly.oura.etu/
45    abal

In [12]:
# Apply
df = data_cleaning.split_actor_binome(df)

# After
df[['actor','binome']]

,actor,binome
0,abaly.oura.etu,
1,abaly.oura.etu,
2,abaly.oura.etu,
3,abaly.oura.etu,
4,abaly.oura.etu,
...,...,...
306941,zacharie.deroo.etu,noe.carpentier.etu
306942,zacharie.deroo.etu,noe.carpentier.etu
306943,zacharie.deroo.etu,noe.carpentier.etu
306944,zacharie.deroo.etu,noe.carpentier.etu


In [13]:
df['binome'].unique()

array(['', None, 'abderrahmen.ayadi.etu', 'hazem.hlioui.etu',
       'serigne.cisse.etu', 'ilian.briki.etu',
       'edohson-arnaud.guedou.etu', 'yasser.jad.etu',
       'nils.charfeddine-barbier.etu', 'nassim.belouar.etu',
       'rayane.denni.etu', 'aissatou-bobo.diallo.etu',
       'fadil.sani-labo.etu', 'anas.rsmouki.etu',
       'romann.szczepaniak.etu', 'mamadou-bachir.barry.etu',
       'sadia.muhammad.etu', 'hajar.amnay.etu', 'nayssam.benessalah.etu',
       'ismail.nejjar.etu', 'alpha-abdoulaye.diallo.etu',
       'thierno-abdoulaye.diallo2.etu', 'ghizlane.aissaoui.etu',
       'avsin.ata.etu', 'julien.buseyne.etu', 'igor.belyaev.etu',
       'elias.debab.etu', 'hamza.chebbah.etu', 'clement.sorge.etu',
       'antoine.bollen.etu', 'pierrick.cardoso.etu', 'mayas.bouhadda.etu',
       'safa-khawthar.draouche.etu', 'hugo.vandewalle2.etu',
       'youssef.airaki.etu', 'anass.saidi.etu', 'mohammed.abdennour.etu',
       'nolan.halluin.etu', 'mervin.kalistas.etu', 'boris.mutagoma.et

#### 5. Remove actors or binomes name

In [14]:
# Before in actor
count = (df['actor'] == 'nebut').sum()
print(f"The value appears {count} times for nebut.")

# Before in binome
count = (df['binome'] == 'luc').sum()
print(f"The value appears {count} times for luc.")

df = data_cleaning.delete_name_actor_binome(df, 'actor', 'nebut')
df = data_cleaning.delete_name_actor_binome(df, 'binome', 'luc')
    
# After in actor
count = (df['actor'] == 'nebut').sum()
print(f"The value appears {count} times for nebut.")

# After in binome
count = (df['binome'] == 'luc').sum()
print(f"The value appears {count} times for luc.")

The value appears 0 times for nebut.
The value appears 14 times for luc.
The value appears 0 times for nebut.
The value appears 0 times for luc.


#### 6. Replace None value by empty string

In [15]:
# Before
print(f"Total value of None in actor: {len(df[df['actor'].isna()])}")
print(f"Total value of None in binome: {len(df[df['binome'].isna()])}")

# Apply
df = data_cleaning.replace_None_by_str(df,'binome')

# After
print("After")
print(f"Total value of None in actor: {len(df[df['actor'].isna()])}")
print(f"Total value of None in binome: {len(df[df['binome'].isna()])}")

Total value of None in actor: 0
Total value of None in binome: 3756
After
Total value of None in actor: 0
Total value of None in binome: 0


#### 7. Replace joker

In [16]:
# Before
print(len(df[df['binome'] == 'MI1304']), len(df[df['binome'] == 'MI1301']))

jokers_real_name = {
    'MI1304': 'mariama-sere.sylla.etu',
    'MI1301': 'mariama-sere.sylla.etu'
}

# Apply
df = data_cleaning.replace_jokers(df, ['actor','binome'],jokers_real_name)

# After
print("After")
print(len(df[df['binome'] == 'MI1304']), len(df[df['binome'] == 'MI1301']))

52 445
After
0 0


#### 8. Cleaning manually

In [17]:
# Before
print(len(df[df['actor'] == 'anis.younes.etu']))

# Apply
df = data_cleaning.cleaning_manual_actors_2425(df, 'anis.younes.etu')

# After
print("After")
print(len(df[df['actor'] == 'anis.younes.etu@univ-lille.fr']))


2
After
0


In [18]:
# Retry the test to see the number of incorrects
incorrect = data_cleaning.not_a_correct_identifier(df)
incorrect

Series([], dtype: object)

In [19]:
# Save new df
io_utils.write_csv(df,INTERIM_DATA_DIR)

Directory is ok.
File saved.


### Read clean dataframe

In [20]:
df = io_utils.reading_dataframe(dir= INTERIM_DATA_DIR, file_name='actuer_nettoyage_2425.csv')

Directory is ok.


In [21]:
df['actor'].unique()

array(['abaly.oura.etu', 'abdallah.saint-ghislain.etu',
       'abdelrahmane.bendjeladjel.etu', 'abderrahmen.ayadi.etu',
       'abdoulaye.mbaye.etu', 'abdoulaye.nguere.etu',
       'adam.chikh-touami.etu', 'adam.oukzedou.etu', 'adam.sialiti.etu',
       'adama-eliya.avlah.etu', 'adame.chairi.etu', 'adem.kaloun.etu',
       'adnane.el-mazouni.etu', 'adrien.vetter.etu', 'ahmed.bouziani.etu',
       'ahmed.terfas.etu', 'ahmedzeidane.mohamedelmoctar.etu',
       'aissatou-bobo.diallo.etu', 'alban.gahide-fournier.etu',
       'alicia.zouadi.etu', 'alix.carton2.etu',
       'alpha-abdoulaye.diallo.etu', 'alpha-mahmoudou.diallo.etu',
       'amadou-madani.balde.etu', 'amadou.balde.etu', 'amel.hadjloum.etu',
       'amine.khadri.etu', 'anaba.hilary-williams.etu',
       'anaelle.brient.etu', 'anaelle.case.etu', 'anais.bilek.etu',
       'anas.bouchlaghem.etu', 'anas.en-nor.etu', 'anas.rsmouki.etu',
       'anass.saidi.etu', 'andy.ba.etu', 'ania.belaida.etu',
       'anis.bitari.etu', 'anis.me

### Clean **Filename** Field 
Process:
1. Replace None values in filename by ""
2. Split filename, extract the filename
3. Fill empty filename for each verbe (like Run.Program)

#### 1. Replace None values in filename by ""

In [22]:
#Before 
print(df['filename'].isnull().sum())

# Apply 
df = data_cleaning.replace_None_by_str(df,'filename')

#After
print(df['filename'].isnull().sum())

151183
0


#### 2. Split filename, extract the filename

In [23]:
# Before
df['filename'].head(50)

0                                                      
1       /home/l1/abaly.oura.etu/Downloads/puissance4.py
2                                                      
3                                                      
4            /home/l1/abaly.oura.etu/TDM 8/devinette.py
5            /home/l1/abaly.oura.etu/TDM 8/devinette.py
6                                                      
7     /home/l1/abaly.oura.etu/TDM 8/erreurs_boucles_...
8            /home/l1/abaly.oura.etu/TDM 8/devinette.py
9            /home/l1/abaly.oura.etu/TDM 8/devinette.py
10           /home/l1/abaly.oura.etu/TDM 8/devinette.py
11           /home/l1/abaly.oura.etu/TDM 8/devinette.py
12           /home/l1/abaly.oura.etu/TDM 8/devinette.py
13                                                     
14                                                     
15               /home/l1/abaly.oura.etu/TDM 8/while.py
16                                                     
17    /home/l1/abaly.oura.etu/Documents/iterable

In [24]:
# Apply
df['filename'] = data_cleaning.extract_filename(df['filename'])

In [25]:
# After
df['filename'].head(50)

0                             
1                puissance4.py
2                             
3                             
4                 devinette.py
5                 devinette.py
6                             
7     erreurs_boucles_while.py
8                 devinette.py
9                 devinette.py
10                devinette.py
11                devinette.py
12                devinette.py
13                            
14                            
15                    while.py
16                            
17            iterables_for.py
18                            
19                            
20            iterables_for.py
21      iterable_indexation.py
22                            
23                            
24                            
25                            
26                            
27                            
28                            
29                            
30                            
31                     ytdd.py
32      

#### 3. Fill empty filename for Run.Test

In [26]:
df[df['verb']=='Run.Test']['filename'].unique()

array(['devinette.py', 'ytdd.py', 'iterable_indexation.py',
       'experimentations_fichiers.py', 'puissance4.py',
       'erreurs_boucles_while.py', 'iterables_for.py', 'tictactoe.py',
       'fichiers.py', 'erreurs_aliasing.py',
       'erreurs_boucles_while_suite.py', 'parcours_interrompu.py',
       'erreurs_cond().py', 'conditionnels.py', 'manipulations.py',
       'OURATDM3.py', 'erreurs_cond.py', 'conditionnelles.py',
       'erreurs_cond(1).py', 'categories.py', 'pendu.py', 'fichier.py',
       'while.py', 'imprimerie.py', 'jeu_421.py', 'TP-PROG.py',
       'erreurs_multiples.py', 'booleens.py', 'pour_aller_plus_loin.py',
       'complementaire.py', 'suplémentaire.py', 'age.py',
       'controleTPi.py', 'ab.py', 'abderrahmen ayadi prog.py',
       'programation.py', 'prog.py', 'erreur.py', 'asba.py',
       'fonctions.py', 'exemple_simple.py', 'vzvzfzfzz.py',
       'echappement.py', 'interactions_mystere.py', 'premier_script.py',
       'tuyt.py', 'fonctions .py', 'pour_debog

In [27]:
print(df[df['verb']=='Run.Test']['filename'].isna().sum())
print((df[df['verb']=='Run.Test']['filename'] == '').sum())

0
0


There is no Nan or empty strings, OK for filename of Run.Test

#### 3. Fill empty filename for Run.Program

In [28]:
# Before
total       = len(df[df['verb']=='Run.Program'])
total_nan   = df[df['verb']=='Run.Program']['filename'].isna().sum()
total_empty = (df[df['verb']=='Run.Program']['filename'] == '').sum()

print(f"Total rows of Run.Program                  : {total}")
print(f"Total rows of Nan in Run.Program           : {total_nan}")
print(f"Total rows of empty strings in Run.Program : {total_empty}")

Total rows of Run.Program                  : 54352
Total rows of Nan in Run.Program           : 0
Total rows of empty strings in Run.Program : 54352


All values of filename for the rows where verb == Run.Program are an empty string.

In [29]:
# Fill by commandRan
mask = df['verb'] == 'Run.Program'
df.loc[mask, 'filename'] = data_cleaning.extract_filename_from_commandRan_Run_Program(df.loc[mask, 'commandRan'])

In [30]:
# After
total_empty = (df[df['verb']=='Run.Program']['filename'] == '').sum()

print(f"Total rows of empty strings in Run.Program : {total_empty}")

Total rows of empty strings in Run.Program : 5791


The number of empty strings reduced from 54352 to 5791 thanks to commandRan.

In [31]:
df[df['verb']=='Run.Program'][['filename','commandRan']].head(50)

,filename,commandRan
22,,%Run -c $EDITOR_CONTENT\n
23,,%Run -c $EDITOR_CONTENT\n
24,,%Run -c $EDITOR_CONTENT\n
25,,%Run -c $EDITOR_CONTENT\n
26,,%Run -c $EDITOR_CONTENT\n
27,,%Run -c $EDITOR_CONTENT\n
28,,%Run -c $EDITOR_CONTENT\n
29,,%Run -c $EDITOR_CONTENT\n
30,,%Run -c $EDITOR_CONTENT\n
45,ytdd.py,%Run ytdd.py\n


In [32]:
# Fill with P_codeState for the rest.
mask = (df['verb'] == 'Run.Program') & (df['filename'] == '')
df.loc[mask, 'filename'] = df.loc[mask, 'P_codeState'].map(data_cleaning.extract_filename_from_P_codestate_Run_Program)

In [33]:
# After
total_empty = (df[df['verb']=='Run.Program']['filename'] == '').sum()

print(f"Total rows of empty strings in Run.Program : {total_empty}")

Total rows of empty strings in Run.Program : 4623


Empty strings reduced from 5791 to 4623 thanks to P_codeState.

In [34]:
df[df['verb']=='Run.Program'][['filename','P_codeState']].head(50)

,filename,P_codeState
22,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
23,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
24,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
25,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
26,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
27,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
28,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
29,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
30,premieres_occurrences.py,"def premieres_occurrences(s:str):\n """""" pre..."
45,ytdd.py,"def premieres_occurrences(s:str):\n """""" pre..."


In [35]:
# Test for values with different pattern or empty string in filename of Run.Program
pattern = re.compile(r'^[\w\-]+\.py$') # Pattern : name(any number or space).py

filenames       = df[df['verb']=='Run.Program']['filename']
invalid_indices = filenames[~filenames.apply(lambda val: isinstance(val, str)  and bool(pattern.fullmatch(val.strip())))].index # Extract invalid index

mask = (df['verb'] == 'Run.Program') & (df['filename'] == '')

invalid_indices = set(invalid_indices) - set(df.loc[mask,'filename'].index) # Remove empty strings

print(f"Total rows    : {len(df[df['verb']=='Run.Program']['filename'])}")
print(f"Invalid rows  : {len(invalid_indices)}")
print(f"Empty strings : {len(df.loc[mask,'filename'])}")

Total rows    : 54352
Invalid rows  : 3643
Empty strings : 4623


From 54352 rows of Run.program, there are 3643 rows which doesn't have the same pattern as name.py in filename; and 4623 rows are empty string in filename.

In [36]:
# Check what are these 3643, first use unique() on their filename to reduces the number of redundent values.
unique_invalid_value = df.loc[list(invalid_indices)]['filename'].unique()
len(unique_invalid_value), unique_invalid_value

(236,
 array(["'itérables.py'", "'renforcé recherche1.py'",
        "'parcours_interrompu (1).py'", "'booléens.py'",
        "'Option RR 1.py'", "'jeu- du- tictactoe.py'",
        "'iterable_indexation (3) (2).py'", "'RR 22_11.py'",
        "'jeu tic tac toe.py'", "'puissance 4.py'", "'tictactoe (1).py'",
        "'Ex de Programmation Conditionnelles.py'", "'booléen.py'",
        'l = [1,2,3,4,5,6,7,8,9].py',
        'create_ui:\n    ui = " | | \\n"\n    print.py',
        "'binairo (2).py'", "'Option RR 29nov.py'",
        'Partie_programmation.py.py', "'Tic tac toe_V2.py'",
        'experimentations_fichiers.py.', "'categories (1).py'",
        "iterables_for.pycompte_car('abracadabra', 'a', 'r')",
        'postfix.lexer.py', "'iterables_for (1).py'",
        "'Programmation3$.py'", 'recette.md', "'TP Fonction.py'",
        "'pour_debogueur (1).py'", "'calcul_interets (2).py'",
        "'Convertion binaire.py'", "'puissance4(1).py'",
        "'booleens (1).py'", "'tictactoe (1) (1).p

There are 236 values not in the exactly pattern (some of them are correct).

In [37]:
# Check empty strings
#pd.set_option('display.max_rows', None) # To show all rows remove the '#'
#pd.set_option('display.max_columns', None) # To show all rows remove the '#'
df.loc[mask,['filename','commandRan','P_codeState']]

,filename,commandRan,P_codeState
832,,%Run -c $EDITOR_CONTENT\n,"# len(""La mer."")\n# max(1, len(""abc""), 123+456..."
833,,%Run -c $EDITOR_CONTENT\n,"# len(""La mer."")\n# max(1, len(""abc""), 123+456..."
1054,,%Run -c $EDITOR_CONTENT\n,NaN
1055,,%Run -c $EDITOR_CONTENT\n,NaN
1056,,%Run -c $EDITOR_CONTENT\n,NaN
...,...,...,...
306902,,%Run -c $EDITOR_CONTENT\n,"prenom = ""Noe""\nlogiciel = ""VSCODE"""
306903,,%Run -c $EDITOR_CONTENT\n,NaN
306904,,%Run -c $EDITOR_CONTENT\n,# Équipe pédagogique de ODI\n# Ce script réali...
306909,,%Run -c $EDITOR_CONTENT\n,# Équipe pédagogique de ODI\n# Ce script réali...


Empty strings without commandRan or P_codeState

#### 3. Fill empty filename for Run.Debugger

In [38]:
# Before
total       = len(df[df['verb']=='Run.Debugger'])
total_nan   = df[df['verb']=='Run.Debugger']['filename'].isna().sum()
total_empty = (df[df['verb']=='Run.Debugger']['filename'] == '').sum()

print(f"Total rows of Run.Debugger                  : {total}")
print(f"Total rows of Nan in Run.Debugger           : {total_nan}")
print(f"Total rows of empty strings in Run.Debugger : {total_empty}")

Total rows of Run.Debugger                  : 9500
Total rows of Nan in Run.Debugger           : 0
Total rows of empty strings in Run.Debugger : 9500


In [41]:
# Fill by commandRan
mask = df['verb'] == 'Run.Debugger'
df.loc[mask, 'filename'] = data_cleaning.extract_filename_from_commandRan_Run_Debugger(df.loc[mask, 'commandRan'])

In [42]:
# After
total_empty = (df[df['verb']=='Run.Debugger']['filename'] == '').sum()

print(f"Total rows of empty strings in Run.Debugger : {total_empty}")

Total rows of empty strings in Run.Debugger : 0


The number of empty strings reduced from 9500 to zero.

In [ ]:
# Check their value
df[df['verb']=='Run.Debugger']['filename'].unique()

array(['iterable_indexation.py', 'erreurs_boucles_while.py',
       'erreurs_aliasing.py', 'erreurs_boucles_while_suite.py',
       'conditionnels.py', 'erreurs_multiples.py', 'erreurs_cond.py',
       'doctest_parser.py', 'categories.py', 'parcours_interrompu.py',
       'pendu.py', 'fichier.py', 'conditionnelles.py', 'jeu_nim.py',
       'pour_debogueur.py', 'booleens.py', 'while.py', 'iterables_for.py',
       'complementaire.py', 'debogueur-for.py', 'devinette.py', 'prog.py',
       'fonctions.py', 'vzvzfzfzz.py', 'echappement.py',
       'interactions_mystere.py', 'manipulations.py', "'fonctions .py'",
       'calcul_interets.py', 'pour_debogeur.py', 'kk.py',
       'exo_complementaire.py', 'elements_consecutifs.py', 'brouillon.py',
       "'TP booleen.py'", "'premier_programme.py .'",
       "'TP1 Bis deboggeur.py'", "'debogueur TP1 bis 2.py'",
       "'debogueur TP2.py'", 'experimentations_fichiers.py.', 'tp8.py',
       'tp7.py', "'tp9 (2).py'", 'tp4.py', 'tp9.py', 'imbriquees.

In [45]:
# Test for values with different pattern in filename of Run.Debugger
pattern = re.compile(r'^[\w\-]+\.py$') # Pattern : name(any number or space).py

filenames       = df[df['verb']=='Run.Debugger']['filename']
invalid_indices = filenames[~filenames.apply(lambda val: isinstance(val, str) and bool(pattern.fullmatch(val.strip())))].index # Extract invalid index

print(f"Total rows    : {len(df[df['verb']=='Run.Debugger']['filename'])}")
print(f"Invalid rows  : {len(invalid_indices)}")

Total rows    : 9500
Invalid rows  : 402


In [46]:
unique_invalid_value = df.loc[list(invalid_indices)]['filename'].unique()
len(unique_invalid_value), unique_invalid_value

(70,
 array(["'fonctions .py'", "'TP booleen.py'", "'premier_programme.py .'",
        "'TP1 Bis deboggeur.py'", "'debogueur TP1 bis 2.py'",
        "'debogueur TP2.py'", 'experimentations_fichiers.py.',
        "'tp9 (2).py'", "'td sem6.py'", "'debogueur odi.py'",
        "'tictactoe (1).py'", "'parcours_interrompu (1).py'",
        'pour.deblogueur', "'while (1).py'", "'Option RR 1.py'",
        "'binairo (3).py'", "'binairo fini.py'", "'puissance 4.py'",
        "'Exercices_de_manipulation_Type_booléen.py'",
        "'TP 11 fichiers.py'", "'booleens (1).py'",
        "'iterable_indexation (1).py'", "'puissance 4 version douae.py'",
        "'chaine de charactere.py'", "'TD-Exercices-supplémentaires.py'",
        "'exercice complementaire iterable.py'", "'jeu gain.py'",
        "'corrigé imbriquée.py'", "'manipulations (1).py'",
        "'puissance4 (1).py'", "'premier_ script.py'",
        "'Ex de manipulation Conditionnelles.py'", "'jeu_nim (2).py'",
        "'Exercice manipulation

There is no empty string, and from 402, there are actually 70 values (remove the redundents), they are in the format name.py but they are maybe correct.

In [50]:
# Until now how many filename left
total       = len(df)
total_nan   = df['filename'].isna().sum()
total_empty = (df['filename'] == '').sum()

print(f"Total rows of df                        : {total}")
print(f"Total rows of Nan in filename           : {total_nan}")
print(f"Total rows of empty strings in filename : {total_empty}")

total_empty = (df['filename'] == '').sum()
total_empty

Total rows of df                        : 306914
Total rows of Nan in filename           : 0
Total rows of empty strings in filename : 91954


91954

## Anonymize Data

In [39]:
#anonymized_df = df[['actor', 'binom']].copy() 
#anonymized_df['anonymized_actor'] = df.apply(data_anonymization.anonymize_actor, axis=1)

Replace column actor and binom by anonymized_actor

In [40]:
#df[['actor', 'binom']] = df.apply(data_anonymization.anonymize_actor)


# TODO
- Finish filename cleaning
- Decide what to do with empty strings of Run.Program
- Calculate the total name of student (after cleaning process)
- init_data.py (look at it and the readme of Thomas)
- Anonymization : actor, binom, P-code state, F-code state, columns_with_path
- Process raw data
- Add column that trace the output ?? I don't remember
- Look at notebooknettoyage of Thomas to add a column
- Correct Readme